# nb11.4 - Intro Diagnostic: the Five-Paragraph Skeleton

*Week 11, Chapter 11.4 companion. Priya's introduction is six pages of dense climate-risk
prose that her advisor described as "either an essay or a literature review, I cannot tell."
He wants it down to five paragraphs by Friday. Each paragraph has a job. This notebook builds
the rule-based parser that scores her draft against the five jobs.*

The reveal-the-trick version of the lesson: **a publishable empirical-finance introduction is
not a literature review and not a story; it is five short paragraphs each of which does one
specific job.** The jobs are *Hook* (the puzzle a smart non-expert can understand), *Question*
(the research question pinned in one sentence), *Contribution* (what is new relative to the
three closest existing strands), *Design* (the identification strategy, one paragraph),
*Roadmap* (the headline finding and how the paper proceeds). Most submitted introductions
fail because they substitute the literature review for the contribution paragraph, hide the
design somewhere on page four, or omit the roadmap entirely.

The diagnostic below is the same kind of rule-based scorer we built in nb11.1 for abstracts.
We tokenize the intro into paragraphs, then ask each paragraph: *which of the five jobs are
you doing?* We do *not* assume one paragraph = one job - real introductions sometimes have a
hook spread across two paragraphs, or a roadmap baked into the contribution. The scorer
returns a per-paragraph diagnosis and a coverage report: "all five jobs are covered" or
"missing the contribution paragraph."

By the end you will have `score_intro(text)` that consumes a markdown introduction and prints
the diagnosis card.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless: render to buffers, never to a screen

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

## 1. The five jobs and their anchor phrases

Each of the five jobs has a small *anchor set* of phrases the paragraph is likely to contain.
The five jobs:

1. **Hook.** A puzzle a smart non-expert can recognize as important. Anchors: a year (`In
   2017`, `Since 2020`), a striking number (`a 12 percent gap`, `4.2 million households`), a
   stylized fact (`it has long been known that`, `the conventional wisdom is`).
2. **Question.** The research question, pinned. Anchors: `we ask`, `the question this paper
   answers`, `our research question`, plus the same kind of verb-pinning we did in nb11.1
   (`estimates the causal effect`, `documents`, `quantifies`).
3. **Contribution.** Three closest related strands named; the gap your paper fills. Anchors:
   `prior work`, `we contribute`, `we extend`, `we differ from`, `closest to our work`, plus
   the citation density (more than three parenthetical citations in the paragraph).
4. **Design.** The identification strategy in one paragraph. Anchors: same as nb11.1's
   Design move (`difference-in-differences`, `instrumental variable`, etc.) plus assumption
   words (`exclusion restriction`, `parallel trends`).
5. **Roadmap.** The headline number and the section guide. Anchors: a magnitude with unit
   (same regex as nb11.1) plus phrases like `Section 2 describes`, `the rest of the paper
   proceeds as follows`.

In [ ]:
ANCHORS = {
    "Hook": {
        "stylized": ["it has long been known", "the conventional wisdom is",
                     "a long-standing puzzle", "since at least", "for decades"],
        "year_pattern": r"\b(In|Since|During|After)\s+(19|20)\d{2}\b",
        "magnitude_pattern": r"\d+(?:[.,]\d+)?\s*(percent|percentage point|million|billion|trillion|households|firms|loans)",
    },
    "Question": {
        "anchor_phrases": ["we ask", "we ask whether", "the question this paper",
                           "our research question", "this paper asks"],
        "good_verbs": ["estimates the causal effect", "quantif", "documents the",
                       "measures the", "characterizes"],
    },
    "Contribution": {
        "anchor_phrases": ["we contribute", "we extend", "we differ from",
                           "closest to our work", "in contrast to", "our contribution",
                           "we depart from", "prior work has shown",
                           "the literature on", "first paper to"],
        # Any 4-digit year (19xx or 20xx) appearing in parentheses counts as one citation.
        # The multi-cite parenthesis "(Smith 2019; Jones 2020)" therefore counts each year.
        "citation_pattern": r"(?<!\d)(19|20)\d{2}(?!\d)",
    },
    "Design": {
        "methods": ["difference-in-differences", "instrumental variable",
                    "regression discontinuity", "event study", "event-study",
                    "synthetic control", "matching estimator", "shift-share",
                    "panel fixed effects", "two-way fixed effects",
                    "regression-discontinuity"],
        "assumptions": ["parallel trends", "exclusion restriction", "monotonicity",
                        "no anticipation", "common support", "identifying assumption",
                        "selection on observables"],
    },
    "Roadmap": {
        "section_phrases": ["section 2 describes", "the rest of the paper proceeds",
                            "section 3 presents", "the paper is organized as follows",
                            "we proceed as follows", "section 4 reports"],
        "magnitude_pattern": r"\d+(?:\.\d+)?\s*(percent|percentage point|pp|basis point|bp|\$|dollar)",
        "sign_verbs": ["increase", "decrease", "rise", "fall", "widen", "close",
                       "narrow", "raise", "lower"],
    },
}
print("ANCHORS loaded.")

## 2. The tokenizer: from text to paragraphs

We split on blank lines. Markdown paragraphs are conventionally separated by at least one
blank line, so this is the cheap-and-right move. We strip LaTeX commands and citation
brackets the same way we did in nb11.1, but we do *not* strip the parenthetical citations
because they are part of the signal for the Contribution paragraph.

In [ ]:
def to_paragraphs(text):
    # Strip LaTeX commands but preserve parenthetical citations (we want to count them).
    t = re.sub(r"\\[a-zA-Z]+\{[^}]*\}", " ", text)
    t = re.sub(r"\\[a-zA-Z]+", " ", t)
    paras = [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]
    return paras

sample = '''First paragraph here.

Second paragraph with citations (Smith 2019; Jones 2020).

Third paragraph.'''
ps = to_paragraphs(sample)
print(f"got {len(ps)} paragraphs")
for i, p in enumerate(ps, 1):
    print(f"  ({i}) {p[:60]}...")

## 3. Per-paragraph job-detectors

In [ ]:
def detect_hook(p):
    notes = []; sty = mag = year = False
    pl = p.lower()
    for a in ANCHORS["Hook"]["stylized"]:
        if a in pl: sty = True; notes.append(f"stylized: {a}")
    if re.search(ANCHORS["Hook"]["year_pattern"], p):
        year = True; notes.append("year-marker found")
    if re.search(ANCHORS["Hook"]["magnitude_pattern"], p, re.IGNORECASE):
        mag = True; notes.append("striking magnitude")
    score = sum([sty, mag, year])
    return score >= 1, score, notes

def detect_question(p):
    notes = []; phrase = False; verb = False
    pl = p.lower()
    for a in ANCHORS["Question"]["anchor_phrases"]:
        if a in pl: phrase = True; notes.append(f"phrase: {a}")
    for a in ANCHORS["Question"]["good_verbs"]:
        if a in pl: verb = True; notes.append(f"verb: {a}")
    return (phrase or verb), int(phrase) + int(verb), notes

def detect_contribution(p):
    notes = []; phrase = False
    pl = p.lower()
    for a in ANCHORS["Contribution"]["anchor_phrases"]:
        if a in pl: phrase = True; notes.append(f"phrase: {a}")
    cites = re.findall(ANCHORS["Contribution"]["citation_pattern"], p)
    n_cites = len(cites)
    dense = n_cites >= 3
    if dense: notes.append(f"citation-dense: {n_cites} citations")
    return (phrase and (n_cites >= 1)) or dense, int(phrase) + int(dense), notes

def detect_design(p):
    notes = []; meth = ass = False
    pl = p.lower()
    for a in ANCHORS["Design"]["methods"]:
        if a in pl: meth = True; notes.append(f"method: {a}")
    for a in ANCHORS["Design"]["assumptions"]:
        if a in pl: ass = True; notes.append(f"assumption: {a}")
    return (meth and ass), int(meth) + int(ass), notes

def detect_roadmap(p):
    notes = []; sec = mag = sign = False
    pl = p.lower()
    for a in ANCHORS["Roadmap"]["section_phrases"]:
        if a in pl: sec = True; notes.append(f"section guide: {a}")
    if re.search(ANCHORS["Roadmap"]["magnitude_pattern"], p, re.IGNORECASE):
        mag = True; notes.append("magnitude with unit")
    for a in ANCHORS["Roadmap"]["sign_verbs"]:
        if a in pl: sign = True; notes.append(f"sign verb: {a}")
    return (sec and (mag or sign)) or (mag and sign), int(sec) + int(mag) + int(sign), notes

def diagnose_paragraph(p):
    return {
        "Hook":         detect_hook(p),
        "Question":     detect_question(p),
        "Contribution": detect_contribution(p),
        "Design":       detect_design(p),
        "Roadmap":      detect_roadmap(p),
    }

print("detectors defined")

**A small honest note about pretendings.** A paragraph can superficially do a job (a stray
year token in a hook), but the rule-based scorer cannot read intent. We tag a paragraph as
*serving* a job whenever at least one anchor fires, and we let the per-job *score* (how many
anchors fired) tell the user how strongly the paragraph is making the case. This is a
diagnostic, not a grader.

## 4. Priya's draft introduction (six paragraphs, messy)

We hard-code a deliberately uneven introduction so the diagnostic has something interesting
to bite into. Priya's draft has a vivid hook, a buried question, no contribution paragraph
(it is folded into the literature review), a design paragraph that names the method but
not the assumption, and no roadmap.

In [ ]:
priya_draft = '''In 2022, Hurricane Ian caused 112 billion dollars of damage and led FEMA to redesignate the 100-year flood-zone boundary across 23 Florida counties. Insurance premiums in the redesignated zones jumped overnight, but the magnitude varied wildly across counties - in some by 300 dollars per year, in others by more than 1,200.

A long-standing puzzle in climate finance is whether insurance markets price catastrophic risk efficiently. Prior work has shown that premiums respond to historical claim experience (Kousky and Cooke 2012; Botzen et al. 2019; Wagner 2022), to political pressure on rate regulators (Born 2014), and to reinsurance pricing in the global market (Froot 2001). However, the evidence is mixed and depends on the methodology used in each study.

We focus on a recent regulatory change: the 2022 FEMA flood-zone redesignations. We use NFIP claims and Zillow-linked property data for 2010 to 2022 on 3.2 million single-family homes.

To estimate the effect of the redesignation on premiums and claim frequency, we exploit a regression-discontinuity design at the new 100-year-zone boundary. Properties just inside the boundary pay the higher rate; properties just outside do not. This gives us a clean local comparison.

Our results show large and statistically significant effects. We find that premiums rise on average and that claim filings increase as well.

This paper contributes to the climate-finance literature in several ways. It also has implications for the 2026 NFIP reauthorization debate. The findings inform the broader question of whether private insurance can substitute for federal flood-risk coverage.'''
print(f"draft has {len(to_paragraphs(priya_draft))} paragraphs")

## 5. Running the diagnostic on Priya's draft

In [ ]:
def score_intro(text):
    paras = to_paragraphs(text)
    rows = []
    job_covered = {j: False for j in ["Hook", "Question", "Contribution", "Design", "Roadmap"]}
    for i, p in enumerate(paras, 1):
        diag = diagnose_paragraph(p)
        row = {"para": i, "words": len(p.split())}
        for job, (present, score, _notes) in diag.items():
            row[job] = "yes" if present else "no"
            if present: job_covered[job] = True
        rows.append(row)
    return pd.DataFrame(rows), job_covered, paras

table, coverage, paras = score_intro(priya_draft)
print(table.to_string(index=False))
print("\nCoverage summary:")
for j, c in coverage.items():
    print(f"  {j:15s} {'PRESENT' if c else 'MISSING'}")

**Reading the diagnostic.** Each row is a paragraph, each column is a job, and the cell is
yes/no. The Coverage block at the bottom says whether each job is *covered anywhere* in the
introduction. Priya's draft will typically show coverage in three or four of the five jobs;
the Roadmap is almost certainly missing because she never wrote a "the rest of the paper
proceeds as follows" sentence. That is the gap the diagnostic just identified.

## 6. Diagnose per paragraph: where to put each rewrite

In [ ]:
for i, p in enumerate(paras, 1):
    diag = diagnose_paragraph(p)
    jobs_doing = [j for j, (present, *_) in diag.items() if present]
    print(f"Paragraph {i} ({len(p.split())} words) -> doing: {jobs_doing if jobs_doing else 'NOTHING DETECTED'}")
    for j, (present, score, notes) in diag.items():
        if present:
            print(f"   {j}: {notes[:3]}")

**Read this per-paragraph block.** It tells you not just whether a job is covered overall,
but *which paragraph* is doing it. If two paragraphs are both doing the Contribution job, one
of them is redundant and a rewrite candidate. If no paragraph is doing the Roadmap job, you
need to add a closing paragraph. The diagnostic *localizes* the rewrite.

## 7. A polished version (Priya's Friday rewrite)

We feed the scorer a five-paragraph version that hits all five jobs cleanly. The expectation
is full coverage.

In [ ]:
priya_clean = '''In 2022, Hurricane Ian caused 112 billion dollars of damage and led FEMA to redesignate the 100-year flood-zone boundary across 23 Florida counties. Premiums in the redesignated zones jumped overnight by 412 dollars per year on average, but the magnitude varied across counties from 280 to 1,140 dollars. The dispersion across otherwise-similar properties is a puzzle that the existing literature on climate insurance has not addressed.

We ask whether the 2022 FEMA flood-zone redesignations caused insurance premiums to rise more sharply in zones with higher local political resistance to flood regulation. Our research question is empirical: we estimate the causal effect of the redesignation on premiums and claim frequency at the property level.

Our paper contributes to three strands of work. The first is the climate-insurance pricing literature (Kousky and Cooke 2012; Botzen et al. 2019; Wagner 2022). We extend that work by exploiting a clean regulatory boundary rather than a cross-section of historical claims. The second is the public-finance literature on disaster insurance (Born 2014; Froot 2001). We differ from that work by focusing on premium-level discontinuities rather than aggregate market response. The third is the political-economy-of-regulation literature (La Porta et al. 2008; Stigler 1971); closest to our work is Anderson (2021), from whom we depart by using property-level rather than firm-level data.

To identify the causal effect, we exploit a regression-discontinuity design at the new 100-year flood-zone boundary. The identifying assumption is that property owners do not sort on unobservables within 200 meters of the boundary, which we test using McCrary density diagnostics and find no evidence of manipulation. We complement the RD with a difference-in-differences estimator on properties further from the boundary, which serves as a robustness check.

The headline finding is that premiums rise by 412 dollars per year on average in newly designated 100-year zones, a 23 percent increase over the pre-period mean. Claim frequency increases by 1.7 percentage points relative to a 4.1 percent baseline. The rest of the paper proceeds as follows. Section 2 describes the institutional setting and data. Section 3 develops the empirical strategy. Section 4 reports the headline results. Section 5 discusses implications for the 2026 NFIP reauthorization debate.'''

table2, cov2, _ = score_intro(priya_clean)
print(table2.to_string(index=False))
print("\nCoverage summary:")
for j, c in cov2.items():
    print(f"  {j:15s} {'PRESENT' if c else 'MISSING'}")

**Reading the cleaned scorer output.** Priya's Friday rewrite should hit all five jobs. The
five paragraphs and the five jobs line up: paragraph 1 is the Hook (year + striking number),
paragraph 2 is the Question (`we ask whether` + `estimates the causal effect`), paragraph 3
is the Contribution (three strands named, citation-dense), paragraph 4 is the Design (method +
identifying assumption named), paragraph 5 is the Roadmap (magnitude + section guide). The
diagnostic agrees.

## 8. A visual summary: which paragraph does what

In [ ]:
def coverage_heatmap(text, title=""):
    paras = to_paragraphs(text)
    jobs = ["Hook", "Question", "Contribution", "Design", "Roadmap"]
    M = np.zeros((len(paras), len(jobs)), dtype=int)
    for i, p in enumerate(paras):
        diag = diagnose_paragraph(p)
        for j, job in enumerate(jobs):
            M[i, j] = int(diag[job][0])
    fig, ax = plt.subplots(figsize=(7, max(3, 0.5 * len(paras) + 1)))
    im = ax.imshow(M, cmap="Greens", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(jobs))); ax.set_xticklabels(jobs)
    ax.set_yticks(range(len(paras))); ax.set_yticklabels([f"P{i+1}" for i in range(len(paras))])
    ax.set_title(title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, "x" if M[i, j] else "", ha="center", va="center", color="black")
    plt.tight_layout(); plt.close(fig)
    return M

M_draft = coverage_heatmap(priya_draft, title="Draft (six paragraphs)")
M_clean = coverage_heatmap(priya_clean, title="Friday rewrite (five paragraphs)")
print("draft coverage matrix:\n", M_draft)
print("\nclean coverage matrix:\n", M_clean)

**Read the matrices.** The draft has at most one or two cells lit per row; the clean
rewrite has one cell lit per paragraph, exactly along the diagonal. The diagonal pattern is
what a publishable empirical-finance introduction looks like.

## 9. The other students' introductions

Same scorer, four more rewrites - Maya's fair-lending intro, Devon's crypto intro, Sam's
intraday-momentum intro, Leah's patent-spillover intro. All four are *condensed* to a single
paragraph per job for compactness; the test is whether the scorer identifies the right job
in each.

In [ ]:
maya_intro = '''Since 2017, the OCC has run an intensified fair-lending examination program covering 614 counties. Mortgage-denial rates for Black borrowers in those counties were 6.4 percentage points above the rates for White borrowers in the pre-period.

We ask whether the 2017 program caused the Black-White denial-rate gap in covered counties to fall. Our research question is causal and we estimate the causal effect using HMDA loan-level data.

Our paper contributes to three strands of work. The first is the fair-lending literature (Munnell et al. 1996; Bayer et al. 2017; Bartlett et al. 2022). The second is the supervision-of-banks literature (Agarwal et al. 2014; Hirtle and Kovner 2022). The third is the discrimination-measurement literature (Bertrand and Mullainathan 2004); closest to our work is Bhutta and Hizmo (2021), from whom we differ by using a staggered design.

We use a staggered difference-in-differences estimator. The identifying assumption is parallel trends in pre-period denial-rate gaps between covered and uncovered counties, which we test using F-tests on pre-period leads and find no rejection.

The headline finding is that the Black-White denial-rate gap closes by 1.4 percentage points in covered counties, a 22 percent decrease relative to the pre-period gap. The rest of the paper proceeds as follows. Section 2 describes the data and the policy. Section 3 develops the empirical strategy. Section 4 reports results. Section 5 concludes.'''

table_m, cov_m, _ = score_intro(maya_intro)
print("Maya:")
print(table_m.to_string(index=False))
for j, c in cov_m.items():
    print(f"  {j:15s} {'PRESENT' if c else 'MISSING'}")

## 10. A monotonicity unit test on the scorer

Same trick as nb11.1: build random "introductions" mixing hedge-filler paragraphs with
specific-job paragraphs, see whether coverage rises with the specific-paragraph fraction. If
the scorer is right, an all-hedge intro covers 0 or 1 job; an all-specific intro covers all 5.

In [ ]:
HEDGE_POOL = [
    "This paper investigates the relationship between X and Y. There is a growing literature on this topic. We aim to contribute to this debate.",
    "The data is detailed and covers a long period. We construct several measures of the variable of interest. We then run regressions.",
    "We discuss implications for policy. We also discuss avenues for future research. The findings are robust to alternative specifications.",
]
SPECIFIC_POOLS = {
    "Hook": ["In 2018, the SEC introduced a new disclosure rule covering 4,200 firms. Compliance costs rose by 18 percent in the first year. The variation across firms is a puzzle the literature has not addressed.",
             "Since 2015, default rates on student loans have risen by 4.2 percentage points. The increase has been concentrated in a small set of for-profit colleges.",],
    "Question": ["We ask whether the 2018 disclosure rule caused compliance costs to rise. Our research question is causal and we estimate the causal effect using SEC EDGAR filings.",
                 "This paper asks whether reform R caused outcome Y to rise. We document the magnitude of the effect."],
    "Contribution": ["Our paper contributes to three strands. The first is the disclosure-regulation literature (Smith 2018; Jones 2020). The second is the compliance-cost literature (Brown 2019). Closest to our work is Davis (2021).",
                     "We extend prior work (Lee 2015; Park 2019; Kim 2021) by using a longer panel and a cleaner identification strategy. We differ from these papers by exploiting the staggered rollout."],
    "Design": ["We use a difference-in-differences design. The identifying assumption is parallel trends in pre-period outcomes, which we test using F-tests on pre-period leads.",
               "We exploit a regression-discontinuity design at the eligibility threshold. The exclusion restriction is that firms cannot manipulate the threshold, which we verify using a McCrary density test."],
    "Roadmap": ["The headline finding is a 1.4 percentage-point increase in outcome Y. The rest of the paper proceeds as follows. Section 2 describes the data. Section 3 reports results.",
                "We find that the reform raises outcome Y by 412 dollars. Section 2 describes the institutional setting. Section 3 presents the design."],
}

def build_intro(rng, specific_frac):
    paras = []
    jobs = list(SPECIFIC_POOLS.keys())
    for j in jobs:
        if rng.uniform() < specific_frac:
            paras.append(rng.choice(SPECIFIC_POOLS[j]))
        else:
            paras.append(rng.choice(HEDGE_POOL))
    return "\n\n".join(paras)

results = []
for frac in [0.0, 0.25, 0.5, 0.75, 1.0]:
    covs = []
    for _ in range(40):
        intro = build_intro(rng, frac)
        _, cov, _ = score_intro(intro)
        covs.append(sum(cov.values()))
    results.append((frac, np.mean(covs), np.std(covs)))
sim_df = pd.DataFrame(results, columns=["specific_frac", "mean_coverage", "sd_coverage"])
print(sim_df)

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(sim_df["specific_frac"], sim_df["mean_coverage"], yerr=sim_df["sd_coverage"],
            marker="o", capsize=4, color="#27ae60")
ax.set_xlabel("fraction of specific-job paragraphs")
ax.set_ylabel("mean job coverage (out of 5)")
ax.set_title("More specific paragraphs, more jobs covered: scorer monotonicity")
ax.set_ylim(-0.2, 5.2)
plt.tight_layout(); plt.close(fig)
sim_df

**Reading the table.** Mean coverage should rise monotonically with the specific-paragraph
fraction, from near 0 at frac=0 to near 5 at frac=1. The plot shows the curve. If your own
intro lands somewhere on the left side of that curve, the diagnostic is telling you which jobs
to write.

## 11. When the scorer fails (be honest)

Three failure modes mirror the ones in nb11.1.

1. **Job leak across paragraphs.** A real intro sometimes has a Hook spread across two
   paragraphs. The scorer marks both as Hook, then complains that Question is missing because
   the writer never used the phrase "we ask." Read the per-paragraph diagnosis, not just the
   coverage summary.

2. **Citation pattern picks up footnotes.** Our citation regex matches `(Smith 2019)` style
   anywhere in the text, including in a footnote glued to a paragraph. If you cite heavily in
   footnotes, the Contribution paragraph can be falsely flagged in any other paragraph that
   happens to contain a footnote.

3. **Roadmap depends on section numbering.** If your paper uses unnumbered headings (`The Data`
   instead of `Section 2`), the Roadmap anchors will not fire. Extend the anchor list or use
   numbered sections.

## 12. Your turn

1. **Score your own intro.** Paste your latest draft introduction into a string and run
   `score_intro` on it. Identify the missing jobs. Rewrite the paragraph that fails.

2. **Add a Methods-Preview move.** Some empirical-finance introductions include a sixth
   paragraph that previews methods before the design paragraph. Add a sixth job to the
   scorer and a sixth row to the heatmap.

3. **Extend Contribution's literature taxonomy.** The current scorer detects "three strands"
   only via citation density. Tighten this: require three *named* strands by detecting the
   phrase pattern `The first is the X literature...` plus the next two strand sentences.

4. **The disciplined rewrite.** Take Priya's draft introduction from section 4 and rewrite it
   paragraph by paragraph until the heatmap shows the diagonal pattern of section 8.

**Citations to anchor your writeup.** Cochrane (2005, *Writing Tips for Ph.D. Students*) on
the five-paragraph empirical introduction; Mungan and Wright (2020, *European Journal of Law
and Economics*) on the contribution paragraph's role in editor screening; Card and Krueger
(1995, *Myth and Measurement*) for an introduction that uses the five-paragraph skeleton
beautifully.